In [1]:
import json
import logging
from pathlib import Path

from transformers import BitsAndBytesConfig

from llm_bayesian_reasoning.estimators.true_false_lm_estimator import (
    TrueFalseLLMEstimator,
)
from llm_bayesian_reasoning.pipeline import (
    PipelineConfig,
    build_or_load_index,
    run_pipeline,
)
from llm_bayesian_reasoning.problog_models.problog_models import (
    ProblogAtom,
    ProblogFormula,
)

logging.basicConfig(level=logging.INFO)

/home/mtsatsev/University/MasterThesisDataScienceOnly/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PREPROCESSED_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/preprocessed_data/parsed_test.jsonl"
)

# Load ground truth from QUEST test.jsonl
QUEST_TEST_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/quest_data/test.jsonl"
)
DOCUMENTS_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/index_data/documents.jsonl"
)
INDEX_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/index_data/bm25_index"
)
OUTPUT_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/results/pipeline_results.jsonl"
)

In [3]:
with open(PREPROCESSED_PATH, "r") as f:
    knowledge_base = [json.loads(line) for line in f]
knowledge_base[0]

{'id': 1,
 'query': 'German nonlinear narrative films or based on works by Bret Easton Ellis',
 'parsed': {'atoms': ['{x} is a film',
   '{x} is a German film',
   '{x} is a nonlinear narrative film',
   '{x} is based on a work by Bret Easton Ellis'],
  'logical query': '((({x} is a film) AND ({x} is a German film) AND ({x} is a nonlinear narrative film)) OR (({x} is a film) AND ({x} is based on a work by Bret Easton Ellis)))'},
 'original_query': '<mark>German nonlinear narrative films</mark> or <mark>Films based on works by Bret Easton Ellis</mark>',
 'metadata': {'template': '_ or _',
  'domain': 'films',
  'fluency': ['Mostly Fluent: It has a few errors or it does not sound natural, but I can understand it.',
   'Mostly Fluent: It has a few errors or it does not sound natural, but I can understand it.',
   'Fluent: It is clear, and grammatically correct.'],
  'meaning': ['Different Meaning: The queries ask for different sets of items. One or more of the highlighted clauses in the o

In [4]:
data = {}

for item in knowledge_base:
    parsed = item["parsed"]
    # Normalize {x} -> {X} so ProblogAtom.to_prompt substitution works correctly
    atoms = [ProblogAtom(atom=a.replace("{x}", "{X}")) for a in parsed["atoms"]]
    problog_formula = ProblogFormula(
        formula=parsed["logical query"].replace("{x}", "{X}")
    )
    data[item["id"]] = {
        "query": item["query"],
        "atoms": atoms,
        "problog_formula": problog_formula,
    }

print(f"Loaded {len(data)} records")

Loaded 1727 records


In [5]:
data = {k: v for k, v in data.items() if k in range(0, 15)}
data

{1: {'query': 'German nonlinear narrative films or based on works by Bret Easton Ellis',
  'atoms': [ProblogAtom(atom='{X} is a film', probability=0, context=None, problog_atom_format='is_a_film'),
   ProblogAtom(atom='{X} is a German film', probability=0, context=None, problog_atom_format='is_a_german_film'),
   ProblogAtom(atom='{X} is a nonlinear narrative film', probability=0, context=None, problog_atom_format='is_a_nonlinear_narrative_film'),
   ProblogAtom(atom='{X} is based on a work by Bret Easton Ellis', probability=0, context=None, problog_atom_format='is_based_on_a_work_by_bret_easton_ellis')],
  'problog_formula': ProblogFormula(formula='((({X} is a film) AND ({X} is a German film) AND ({X} is a nonlinear narrative film)) OR (({X} is a film) AND ({X} is based on a work by Bret Easton Ellis)))', head='formula', problog_formula_format='formula({X}) :-\n    (((is_a_film({X})) ,\n     (is_a_german_film({X})) ,\n     (is_a_nonlinear_narrative_film({X}))) ; ((is_a_film({X})) ,\n 

In [6]:
ground_truth = {}
with open(QUEST_TEST_PATH, "r") as f:
    for i, line in enumerate(f, start=1):
        record = json.loads(line)
        ground_truth[i] = record["docs"]

In [7]:
retriever = build_or_load_index(
    documents_path=DOCUMENTS_PATH, index_path=INDEX_PATH, batch_size=1000, limit=5000
)
print(f"Index ready — {len(retriever.entities)} documents indexed")

INFO:llm_bayesian_reasoning.pipeline.pipeline:Loading existing BM25 index from /home/mtsatsev/University/MasterThesisDataScienceOnly/llm_bayesian_reasoning/data/index_data/bm25_index


Index ready — 5000 documents indexed


In [8]:
estimator = TrueFalseLLMEstimator.from_pretrained(
    model_name="microsoft/phi-2",
    device="cuda",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
)

INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.40s/it]


In [ ]:
config = PipelineConfig(
    top_n=50,
    top_k=10,
    batch_size=1000,
    index_path=INDEX_PATH,
    output_path=OUTPUT_PATH,
)

output = run_pipeline(
    data=data,
    retriever=retriever,
    estimator=estimator,
    config=config,
    ground_truth=ground_truth,
)

results = output["results"]
metrics = output["metrics"]

print(f"Done. Processed {len(results)} records.\n")
print("=== Metrics ===")
for name, value in metrics.items():
    if name == "num_evaluated":
        print(f"  {name:30s} {int(value)}")
    else:
        print(f"  {name:30s} {value:.4f}")